# Importing Libraties

In [8]:
import boto3
import pandas as pd
import json
import time
import sqlite3


# Initializing Bedrock Client and importing cleaned data

In [9]:
client = boto3.client("bedrock-runtime", region_name="us-west-2")
model_id = "global.anthropic.claude-sonnet-4-20250514-v1:0"



movies_conn = sqlite3.connect("db/Final_movies.db")
sample_df  = pd.read_sql_query("SELECT * FROM Final_movies",  movies_conn)
movies_conn.close()

ratings_conn = sqlite3.connect("db/ratings.db")
ratings_df = pd.read_sql_query("SELECT * FROM ratings", ratings_conn)
ratings_conn.close()




print("Clients initialized.")
sample_df.tail()

Clients initialized.


,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,genres,status,avg_rating,rating_count,user_ratings
95,2055,tt0316356,Open Range,A former gunslinger is forced to take up arms ...,"[{""name"": ""Tig Productions"", ""id"": 335}, {""nam...",2003-08-29,22000000,68296293,139.0,"[{""id"": 37, ""name"": ""Western""}]",Released,4.00,1,"[{""userId"": 564, ""rating"": 4.0}]"
96,7459,tt0811080,Speed Racer,Speed Racer is the tale of a young and brillia...,"[{""name"": ""Village Roadshow Pictures"", ""id"": 7...",2008-05-09,120000000,93945766,135.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 10751, ""...",Released,4.75,2,"[{""userId"": 283, ""rating"": 5.0}, {""userId"": 64..."
97,844,tt0212712,2046,2046 is the sequel to Wong Kar-Wais’ successfu...,"[{""name"": ""Paradis Films"", ""id"": 255}, {""name""...",2004-05-20,12000000,19271312,129.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 18, ""na...",Released,3.00,1,"[{""userId"": 86, ""rating"": 3.0}]"
98,4723,tt0405336,Southland Tales,Set in the futuristic landscape of Los Angeles...,"[{""name"": ""Universal Pictures"", ""id"": 33}, {""n...",2006-05-15,17000000,374743,144.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 878, ""na...",Released,3.90,5,"[{""userId"": 15, ""rating"": 3.5}, {""userId"": 367..."
99,1880,tt0087985,Red Dawn,"It is the mid-1980s. From out of the sky, Sovi...","[{""name"": ""United Artists"", ""id"": 60}, {""name""...",1984-08-10,4200000,38376497,114.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 53, ""nam...",Released,4.00,2,"[{""userId"": 41, ""rating"": 4.0}, {""userId"": 407..."


# Defining Standard Helper Functions

In [10]:
def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any":  {"any": {}},
    }
    if tools:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    response = client.converse(**params)
    parts    = response["output"]["message"]["content"]

    return {
        "parts":       parts,
        "stop_reason": response["stopReason"],
        "text":        "\n".join([p["text"] for p in parts if "text" in p]),
    }

# Creating a Tool Calling Schema for Structured Output Extraction and Batch Processing

In [11]:
movie_enrichment_schema = {
    "toolSpec": {
        "name": "movie_enrichment",
        "description": "Enriches movie data with 5 additional attributes based on the provided movie information.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "movies": {
                        "type": "array",
                        "description": "List of enriched movies",
                        "items": {
                            "type": "object",
                            "properties": {
                                "movieId": {
                                    "type": "integer",
                                    "description": "The unique identifier of the movie"
                                },
                                "sentiment": {
                                    "type": "string",
                                    "enum": ["positive", "negative", "neutral"],
                                    "description": "Sentiment of the movie overview"
                                },
                                "budget_tier": {
                                    "type": "string",
                                    "enum": ["low", "medium", "high", "blockbuster"],
                                    "description": "low (<$10M), medium ($10M-$50M), high ($50M-$100M), blockbuster (>$100M)"
                                },
                                "revenue_tier": {
                                    "type": "string",
                                    "enum": ["flop", "average", "hit", "blockbuster"],
                                    "description": "flop (<$10M), average ($10M-$50M), hit ($50M-$100M), blockbuster (>$100M)"
                                },
                                "production_effectiveness_score": {
                                    "type": "integer",
                                    "description": "Score 0-100 based on ROI (revenue/budget) and avg_rating combined"
                                },
                                "audience_appeal": {
                                    "type": "string",
                                    "enum": ["family", "teen", "adult", "general"],
                                    "description": "Target audience based on overview and genres"
                                }
                            },
                            "required": [
                                "movieId",
                                "sentiment",
                                "budget_tier",
                                "revenue_tier",
                                "production_effectiveness_score",
                                "audience_appeal"
                            ]
                        }
                    }
                },
                "required": ["movies"]
            }
        }
    }
}




# Creating the function for batch calling and enabling the tool

In [12]:
def enrich_batch(batch_df):
    movies_text = ""
    for _, row in batch_df.iterrows():
        movies_text += f"""
        Movie ID  : {row['movieId']}
        Title     : {row['title']}
        Overview  : {row['overview']}
        Budget    : ${row['budget']:,.0f}
        Revenue   : ${row['revenue']:,.0f}
        Avg Rating: {row['avg_rating']:.2f} / 5.0
        Genres    : {row['genres']}
        ---"""

    messages = []
    
    add_user_message(
        messages,
        f"""
        You are a movie data analyst. Analyze the movies below and enrich each one with 5 attributes.

        Guidelines:
        - sentiment                    : Analyze the overview tone (positive/negative/neutral)
        - budget_tier                  : low (<$10M), medium ($10M-$50M), high ($50M-$100M), very high (>$100M)
        - revenue_tier                 : flop (<$10M), average ($10M-$50M), hit ($50M-$100M), blockbuster (>$100M)
        - production_effectiveness_score: 0-100 score. Consider ROI (revenue/budget) and avg_rating together.
                                  High ROI + high rating = 90-100. Low ROI + low rating = 0-20.
        - audience_appeal              : Based on overview and genres (family/teen/adult/general)

        <movie_data>
        {movies_text}
        </movie_data>

        Analyse all {len(batch_df)} movies and call the movie_enrichment tool.
        """
    )

    result = chat(
        messages,
        system="You are a movie data analyst. Always call the movie_enrichment tool with your analysis. Never respond with plain text.",
        tools=[movie_enrichment_schema],
        tool_choice="movie_enrichment",
        temperature=0.0
    )

    for part in result["parts"]:
        if "toolUse" in part:
            return part["toolUse"]["input"]["movies"]

    return []

# Batch Processing for Data enrichment

In [13]:
batch_size    = 10
all_results   = []
total_batches = len(sample_df) // batch_size + (1 if len(sample_df) % batch_size != 0 else 0)

for i in range(0, len(sample_df), batch_size):
    batch    = sample_df.iloc[i:i + batch_size]
    batch_no = i // batch_size + 1

    print(f"Processing batch {batch_no}/{total_batches} (movies {i+1}–{min(i+batch_size, len(sample_df))})...")

    try:
        results = enrich_batch(batch)
        all_results.extend(results)
        print(f" {len(results)} movies enriched")
    except Exception as e:
        print(f" Batch {batch_no} failed: {e}")

    time.sleep(1)

print(f"\nTotal enriched : {len(all_results)} / {len(sample_df)}")

Processing batch 1/10 (movies 1–10)...
 10 movies enriched
Processing batch 2/10 (movies 11–20)...
 10 movies enriched
Processing batch 3/10 (movies 21–30)...
 10 movies enriched
Processing batch 4/10 (movies 31–40)...
 10 movies enriched
Processing batch 5/10 (movies 41–50)...
 10 movies enriched
Processing batch 6/10 (movies 51–60)...
 10 movies enriched
Processing batch 7/10 (movies 61–70)...
 10 movies enriched
Processing batch 8/10 (movies 71–80)...
 10 movies enriched
Processing batch 9/10 (movies 81–90)...
 10 movies enriched
Processing batch 10/10 (movies 91–100)...
 10 movies enriched

Total enriched : 100 / 100


# Viewing the New Column Data

In [14]:
enriched_df = pd.DataFrame(all_results)

print(f"\nShape : {enriched_df.shape}")
print(f"\nNull check:\n{enriched_df.isnull().sum()}")
display(enriched_df.head(10))


Shape : (100, 6)

Null check:
movieId                           0
sentiment                         0
budget_tier                       0
revenue_tier                      0
production_effectiveness_score    0
audience_appeal                   0
dtype: int64


,movieId,sentiment,budget_tier,revenue_tier,production_effectiveness_score,audience_appeal
0,2022,positive,high,blockbuster,75,general
1,27022,positive,blockbuster,blockbuster,70,family
2,88,positive,low,blockbuster,85,teen
3,2486,positive,blockbuster,blockbuster,60,family
4,248,positive,low,flop,45,general
5,703,neutral,low,average,70,adult
6,2620,positive,medium,average,65,adult
7,1369,neutral,medium,blockbuster,65,adult
8,1955,neutral,low,average,75,adult
9,2359,negative,low,average,80,adult


# Adding this data to the original DataFrame

In [15]:
final_df = sample_df.merge(enriched_df, on="movieId", how="left")

print(f"Shape after merge : {final_df.shape}")
print(f"\nNew columns added :")
final_df.head(10)


Shape after merge : (100, 19)

New columns added :


,movieId,imdbId,title,overview,productionCompanies,releaseDate,budget,revenue,runtime,genres,status,avg_rating,rating_count,user_ratings,sentiment,budget_tier,revenue_tier,production_effectiveness_score,audience_appeal
0,2022,tt0280590,Mr. Deeds,"When Longfellow Deeds, a small-town pizzeria o...","[{""name"": ""New Line Cinema"", ""id"": 12}, {""name...",2002-06-28,50000000,171269535,96.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",Released,3.423077,13,"[{""userId"": 23, ""rating"": 4.0}, {""userId"": 232...",positive,high,blockbuster,75,general
1,27022,tt0963966,The Sorcerer's Apprentice,Balthazar Blake is a master sorcerer in modern...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",2010-07-13,150000000,215283742,109.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 12, ""na...",Released,4.333333,3,"[{""userId"": 73, ""rating"": 4.5}, {""userId"": 332...",positive,blockbuster,blockbuster,70,family
2,88,tt0092890,Dirty Dancing,Expecting the usual tedium that accompanies a ...,"[{""name"": ""Great American Films Limited Partne...",1987-08-21,6000000,213954274,100.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10402, ""n...",Released,2.891304,23,"[{""userId"": 33, ""rating"": 3.0}, {""userId"": 67,...",positive,low,blockbuster,85,teen
3,2486,tt0449010,Eragon,"In his homeland of Alagaesia, a farm boy happe...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...",2006-12-14,100000000,249288105,104.0,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""na...",Released,3.000000,1,"[{""userId"": 363, ""rating"": 3.0}]",positive,blockbuster,blockbuster,60,family
4,248,tt0055312,Pocketful of Miracles,"Damon Runyon's fairytale, sweet and funny, is ...","[{""name"": ""Franton Production"", ""id"": 1044}]",1961-12-18,2900000,5000000,136.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",Released,3.153846,13,"[{""userId"": 2, ""rating"": 3.0}, {""userId"": 36, ...",positive,low,flop,45,general
5,703,tt0075686,Annie Hall,"In the city of New York, comedian Alvy Singer ...","[{""name"": ""United Artists"", ""id"": 60}]",1977-04-19,4000000,38251425,93.0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",Released,3.000000,1,"[{""userId"": 19, ""rating"": 3.0}]",neutral,low,average,70,adult
6,2620,tt0090660,Armed and Dangerous,"Dooley, a cop wrongly sacked for corruption, t...","[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",1986-08-15,12000000,15945534,88.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 35, ""nam...",Released,4.333333,3,"[{""userId"": 128, ""rating"": 4.0}, {""userId"": 40...",positive,medium,average,65,adult
7,1369,tt0089880,Rambo: First Blood Part II,John Rambo is released from prison by the gove...,"[{""name"": ""TriStar Pictures"", ""id"": 559}, {""na...",1985-05-21,44000000,300400432,96.0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",Released,2.500000,2,"[{""userId"": 492, ""rating"": 3.0}, {""userId"": 56...",neutral,medium,blockbuster,65,adult
8,1955,tt0080678,The Elephant Man,A Victorian surgeon rescues a heavily disfigur...,"[{""name"": ""Brooksfilms"", ""id"": 5612}]",1980-10-02,5000000,26010864,124.0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 36, ""name...",Released,3.838710,31,"[{""userId"": 15, ""rating"": 4.5}, {""userId"": 23,...",neutral,low,average,75,adult
9,2359,tt0386032,Sicko,Sicko is a Michael Moore documentary about the...,"[{""name"": ""The Weinstein Company"", ""id"": 308}]",2007-05-18,9000000,24538513,123.0,"[{""id"": 99, ""name"": ""Documentary""}]",Released,4.133333,30,"[{""userId"": 83, ""rating"": 4.0}, {""userId"": 119...",negative,low,average,80,adult


# Saving The Enriched Data to the DB

In [16]:
conn = sqlite3.connect("db/enriched_movies.db")
final_df.to_sql("enriched_movies", conn, if_exists="replace", index=False)
conn.close()

# Uploading the Same data into Superbase for easy access later.

In [17]:
import requests
import json

# ── Supabase Config ───────────────────────────────────────────────
SUPABASE_URL = "https://zqvjzukfbfcixxaplzyp.supabase.co"
SUPABASE_KEY = "sb_publishable_LHNy1oE6UxRZtndOBjPoVg_bRaz6_VG"

headers = {
    "apikey":        SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type":  "application/json",
    "Prefer":        "return=minimal"
}

upload_df = final_df.copy()

for col in upload_df.select_dtypes(include=["datetime64"]).columns:
    upload_df[col] = upload_df[col].astype(str)

upload_df = upload_df.where(pd.notnull(upload_df), None)

batch_size  = 10
total       = len(upload_df)
success     = 0

for i in range(0, total, batch_size):
    batch   = upload_df.iloc[i:i+batch_size].to_dict(orient="records")
    response = requests.post(
        f"{SUPABASE_URL}/rest/v1/enriched_movies",
        headers=headers,
        data=json.dumps(batch)
    )
    if response.status_code in [200, 201]:
        success += len(batch)
        print(f"  Uploaded rows {i+1}–{min(i+batch_size, total)}")
    else:
        print(f"  Failed rows {i+1}–{min(i+batch_size, total)} : {response.status_code} {response.text}")

print(f"\nTotal uploaded: {success}/{total}")

  Uploaded rows 1–10
  Uploaded rows 11–20
  Uploaded rows 21–30
  Uploaded rows 31–40
  Uploaded rows 41–50
  Uploaded rows 51–60
  Uploaded rows 61–70
  Uploaded rows 71–80
  Uploaded rows 81–90
  Uploaded rows 91–100

Total uploaded: 100/100
